# Fraud Detection - Model Monitoring

This notebook sets up **ML Observability** for the deployed fraud detection model using Snowflake's native **Model Monitor**. The goal is to detect when the model's inputs or predictions start drifting from what it saw during training — a signal that performance may be degrading.

### What we're implementing

1. **Inference Autocapture**: The SPCS inference service automatically logs every request/response. We create a **Dynamic Table** (`FRAUD_INFERENCE_AUTOCAPTURE`) that refreshes every 10 minutes to materialize this autocaptured data into a queryable format.

2. **Training Baseline**: We snapshot the training data distribution (`FRAUD_MONITORING_BASELINE`) so the monitor has a reference point for "normal" feature distributions.

3. **Model Monitor**: A native Snowflake `MODEL MONITOR` that runs daily, comparing production data against the baseline.

### Supported Drift Metrics

| Metric | Description |
|--------|-------------|
| `JENSEN_SHANNON` | Symmetric, bounded [0,1] measure of distribution divergence |
| `POPULATION_STABILITY_INDEX` | Classic binning-based drift metric; >0.1 moderate, >0.2 significant drift |
| `WASSERSTEIN` | Earth mover's distance — measures "cost" to transform one distribution into another |
| `DIFFERENCE_OF_MEANS` | Simple mean shift detection — good for quick sanity checks |

All metrics support configurable **granularity** (day/week/month) and **segment-based** queries (e.g., drift per customer tier).


Once inference traffic flows through the service, drift metrics populate automatically. View results in **Snowsight → AI & ML → Models → FRAUD_DETECTION_MODEL → Monitors**.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from datetime import datetime, timedelta

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()

DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
WAREHOUSE = "ML_WH"
MODEL_NAME = "FRAUD_DETECTION_MODEL"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
versions_df = model_ref.show_versions()
MODEL_VERSION = versions_df.sort_values('created_on', ascending=False).iloc[0]['name']

print(f"Session ready: {DATABASE}.{SCHEMA} | {WAREHOUSE}")
print(f"Model: {MODEL_NAME}:{MODEL_VERSION}")

In [ ]:
registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model = registry.get_model(MODEL_NAME)
mv = model.version(MODEL_VERSION)
print(f"Model: {MODEL_NAME} ({MODEL_VERSION})")

try:
    services_df = mv.list_services()
    for _, svc in services_df.iterrows():
        print(f"  Service: {svc['name']} - {svc['status']}")
except Exception as e:
    print(f"  Could not list services: {e}")

inference_count = session.sql(f"""
    SELECT COUNT(*) AS CNT FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
""").collect()[0]['CNT']
print(f"Auto-captured records: {inference_count:,}")

## Create Source (Dynamic Table) & Baseline

**Source** — Create a Dynamic Table (`FRAUD_INFERENCE_AUTOCAPTURE`) that flattens semi-structured `INFERENCE_TABLE()` auto-capture data into typed columns: 29 input features extracted from `RECORD_ATTRIBUTES` request data, the prediction from response data, and the capture timestamp. Refreshes every 10 minutes.

**Baseline** — Create `FRAUD_MONITORING_BASELINE` from the Feature Store training data (`FRAUD_TRAINING_DATA`). All columns are cast to `FLOAT` to match the Dynamic Table schema. The monitor will compare production feature distributions against this baseline for drift detection.

In [ ]:
FEATURE_COLS = [
    "ACCOUNT_AGE_DAYS", "CREDIT_SCORE", "AVG_MONTHLY_TRANSACTIONS",
    "AVG_TRANSACTION_AMOUNT", "NUM_CARDS_ON_FILE", "IS_PREMIUM_MEMBER",
    "NUM_PREVIOUS_FRAUD_REPORTS", "D7_AVG_TXN_AMOUNT", "D7_STD_TXN_AMOUNT",
    "D7_MAX_TXN_AMOUNT", "D7_MIN_TXN_AMOUNT", "D7_TOTAL_TXN_AMOUNT", "D7_TXN_COUNT",
    "D7_AVG_TXN_HOUR", "D7_STD_TXN_HOUR", "D7_INTL_TXN_COUNT", "D7_INTL_TXN_RATIO",
    "D7_CARD_PRESENT_RATIO", "TRANSACTION_AMOUNT", "TRANSACTION_HOUR",
    "TRANSACTION_DAY_OF_WEEK", "IS_WEEKEND", "CARD_PRESENT", "IS_INTERNATIONAL",
    "AMOUNT_TO_AVG_RATIO", "AMOUNT_TO_MAX_RATIO", "IS_HIGH_VALUE_TXN",
    "IS_ODD_HOUR", "RISK_SCORE"
]

feature_sql = ",\n".join([
    f"    RECORD_ATTRIBUTES:\"snow.model_serving.request.data.{col}\"::FLOAT AS {col}"
    for col in FEATURE_COLS
])

session.sql(f"""
CREATE OR REPLACE DYNAMIC TABLE {DATABASE}.{SCHEMA}.FRAUD_INFERENCE_AUTOCAPTURE
    TARGET_LAG = '10 minutes'
    WAREHOUSE = {WAREHOUSE}
AS
SELECT
    TIMESTAMP::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP,
    RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::FLOAT AS PREDICTION,
    RECORD_ATTRIBUTES:"snow.model_serving.function.name"::VARCHAR AS FUNCTION_NAME,
{feature_sql}
FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
""").collect()
print(f"Source: FRAUD_INFERENCE_AUTOCAPTURE (Dynamic Table, 10-min refresh)")

feature_casts = [f"{col}::FLOAT AS {col}" for col in FEATURE_COLS]
session.sql(f"""
CREATE OR REPLACE TABLE {DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE AS
SELECT
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP,
    IS_FRAUD::FLOAT AS PREDICTION,
    'predict' AS FUNCTION_NAME,
    {', '.join(feature_casts)}
FROM {DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA
""").collect()
baseline_count = session.table(f"{DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE").count()
print(f"Baseline: FRAUD_MONITORING_BASELINE ({baseline_count:,} rows from training data)")

## Create & Verify Model Monitor

A **Model Monitor** is a first-class Snowflake object — not a Dynamic Table or a view. Think of it as a **managed scheduled task** that:
1. Runs on a configurable interval (e.g., daily)
2. Reads production inference data from a **source** table
3. Compares feature/prediction distributions against a **baseline** table
4. Computes drift, performance, and statistical metrics
5. Stores results internally, queryable via `MODEL_MONITOR_DRIFT_METRIC()` and related functions

It auto-detects the ML task type (binary classification, regression, etc.) from the model metadata.

**Below** — `CREATE MODEL MONITOR` binds the model version, source table, and baseline together. Configured with daily refresh and 1-day aggregation windows. `PREDICTION_SCORE_COLUMNS` tells the monitor which column holds the model output.

In [ ]:
session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {DATABASE}.{SCHEMA}.FRAUD_DETECTION_MONITOR WITH
    MODEL = {MODEL_NAME}
    VERSION = '{MODEL_VERSION}'
    FUNCTION = 'predict'
    SOURCE = {DATABASE}.{SCHEMA}.FRAUD_INFERENCE_AUTOCAPTURE
    WAREHOUSE = {WAREHOUSE}
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = INFERENCE_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTION')
    BASELINE = {DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE
""").collect()

row = session.sql(f"""
    SHOW MODEL MONITORS LIKE 'FRAUD_DETECTION_MONITOR' IN SCHEMA {DATABASE}.{SCHEMA}
""").collect()[0].as_dict()
print(f"Monitor: FRAUD_DETECTION_MONITOR")
print(f"  State: {row.get('monitor_state', 'N/A')}")
print(f"  Task: {row.get('model_task', 'N/A')}")
print(f"  Refresh: {row.get('refresh_interval', 'N/A')} | Window: {row.get('aggregation_window', 'N/A')}")

## Query Drift & Statistical Metrics

**Drift** — `MODEL_MONITOR_DRIFT_METRIC()` returns PSI (Population Stability Index) for a given feature over a time range. PSI < 0.1 = stable, 0.1–0.25 = moderate drift, > 0.25 = significant drift.

**Stats** — `MODEL_MONITOR_STAT_METRIC()` returns count, null rate, and distribution stats per aggregation window. Both return empty results until the source Dynamic Table has inference data.

In [ ]:
try:
    drift_rows = session.sql("""
        SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'FRAUD_DETECTION_MONITOR',
            'POPULATION_STABILITY_INDEX',
            'TRANSACTION_AMOUNT',
            '1 DAY',
            DATEADD('DAY', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
    """).collect()
    if drift_rows:
        for row in drift_rows:
            print(row.as_dict())
    else:
        print("No drift data yet (source empty - will populate with inference traffic)")
except Exception as e:
    print(f"Drift: {e}")

try:
    stat_rows = session.sql("""
        SELECT * FROM TABLE(MODEL_MONITOR_STAT_METRIC(
            'FRAUD_DETECTION_MONITOR',
            'COUNT',
            'PREDICTION',
            '1 DAY',
            DATEADD('DAY', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
    """).collect()
    if stat_rows:
        for row in stat_rows:
            print(row.as_dict())
    else:
        print("No stat data yet")
except Exception as e:
    print(f"Stats: {e}")

In [ ]:
print("=== Model Monitoring Setup Complete ===")
print(f"Source:   {DATABASE}.{SCHEMA}.FRAUD_INFERENCE_AUTOCAPTURE")
print(f"Baseline: {DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE")
print(f"Monitor:  {DATABASE}.{SCHEMA}.FRAUD_DETECTION_MONITOR")
print(f"Model:    {MODEL_NAME} ({MODEL_VERSION})")
print("\nView in Snowsight: AI & ML > Models > FRAUD_DETECTION_MODEL > Monitors")